In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.types import IntegerType
from pyspark.sql.types import FloatType
from pyspark.sql.functions import col

In [2]:
# Create Spark session
spark = SparkSession.builder \
    .appName("PySpark Finance") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/04 18:42:19 WARN Utils: Your hostname, DESKTOP-MKL4RFT, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/04 18:42:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/04 18:42:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("Spark Version:", spark.version)

Spark Version: 4.1.2


### Load Data

In [ ]:
data = spark.read.csv("train.csv", header=True, inferSchema=True)
data.show(5)

+---+----------+--------------+-----------+---------+------+----+------+---------+-------------+---------+--------------+---------------+------+
| id|CustomerId|       Surname|CreditScore|Geography|Gender| Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---+----------+--------------+-----------+---------+------+----+------+---------+-------------+---------+--------------+---------------+------+
|  0|  15674932|Okwudilichukwu|        668|   France|  Male|33.0|     3|      0.0|            2|      1.0|           0.0|      181449.97|     0|
|  1|  15749177| Okwudiliolisa|        627|   France|  Male|33.0|     1|      0.0|            2|      1.0|           1.0|        49503.5|     0|
|  2|  15694510|         Hsueh|        678|   France|  Male|40.0|    10|      0.0|            2|      1.0|           0.0|      184866.69|     0|
|  3|  15741417|           Kao|        581|   France|  Male|34.0|     2|148882.54|            1|      1.0|           1.0|       84

In [7]:
data.printSchema()

root
 |-- id: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: double (nullable = true)
 |-- IsActiveMember: double (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)



In [11]:
data.select("Geography", "Gender", "CreditScore", "Exited").show(10)

+---------+------+-----------+------+
|Geography|Gender|CreditScore|Exited|
+---------+------+-----------+------+
|   France|  Male|        668|     0|
|   France|  Male|        627|     0|
|   France|  Male|        678|     0|
|   France|  Male|        581|     0|
|    Spain|  Male|        716|     0|
|  Germany|  Male|        588|     1|
|   France|Female|        593|     0|
|    Spain|  Male|        678|     0|
|   France|  Male|        676|     0|
|  Germany|  Male|        583|     0|
+---------+------+-----------+------+
only showing top 10 rows


In [12]:
data.where("Exited == 1").select("Geography", "Gender", "CreditScore").show(10)

+---------+------+-----------+
|Geography|Gender|CreditScore|
+---------+------+-----------+
|  Germany|  Male|        588|
|  Germany|Female|        645|
|   France|  Male|        559|
|  Germany|  Male|        554|
|    Spain|  Male|        703|
|    Spain|  Male|        785|
|    Spain|Female|        797|
|  Germany|  Male|        749|
|  Germany|  Male|        551|
|  Germany|  Male|        747|
+---------+------+-----------+
only showing top 10 rows


In [18]:
data.where(data["Exited"] == 1).groupBy("Geography").count().show()

+---------+-----+
|Geography|count|
+---------+-----+
|  Germany|13114|
|   France|15572|
|    Spain| 6235|
+---------+-----+



In [19]:
data.where(data["Exited"] == 0).groupBy("Geography").count().show()

+---------+-----+
|Geography|count|
+---------+-----+
|  Germany|21492|
|   France|78643|
|    Spain|29978|
+---------+-----+



In [41]:
def BalanceSalaryRatio(balanace, estimated_salary):
        return balanace / estimated_salary

In [37]:
def TotalProductsUsed(num_prod, tenure):
    return num_prod * tenure

In [21]:
def GeoGender(geo, gender):
    return f"{geo}_{gender}"

In [51]:
def CardActiveMember(has_card, is_active):
    if has_card == 1 and is_active == 1:
        return "Active"
    elif has_card == 1 and is_active == 0:
        return "Inactive"
    else:
        return "NoCard"

In [50]:
geo_gender_udf = udf(GeoGender, StringType())
balance_salary_ratio_udf = udf(BalanceSalaryRatio, FloatType())
total_products_used_udf = udf(TotalProductsUsed, IntegerType())
card_active_member_udf = udf(CardActiveMember, StringType())

In [52]:
data.select('hasCrCard', 'IsActiveMember').show(10)

+---------+--------------+
|hasCrCard|IsActiveMember|
+---------+--------------+
|      1.0|           0.0|
|      1.0|           1.0|
|      1.0|           0.0|
|      1.0|           1.0|
|      1.0|           1.0|
|      1.0|           0.0|
|      1.0|           0.0|
|      1.0|           0.0|
|      1.0|           0.0|
|      1.0|           1.0|
+---------+--------------+
only showing top 10 rows


In [53]:
data.select('CreditScore', 
            geo_gender_udf('Geography', 'Gender').alias("GeoGender"), 
            balance_salary_ratio_udf('Balance', 'EstimatedSalary').alias("BE_Ratio"),
            total_products_used_udf('NumOfProducts', 'Tenure').alias("TotalProductsUsed"),
            card_active_member_udf('HasCrCard', 'IsActiveMember').alias("CardActiveMember")
            ).show(10)

+-----------+-------------+----------+-----------------+----------------+
|CreditScore|    GeoGender|  BE_Ratio|TotalProductsUsed|CardActiveMember|
+-----------+-------------+----------+-----------------+----------------+
|        668|  France_Male|       0.0|                6|        Inactive|
|        627|  France_Male|       0.0|                2|          Active|
|        678|  France_Male|       0.0|               20|        Inactive|
|        581|  France_Male|  1.760655|                2|          Active|
|        716|   Spain_Male|       0.0|               10|          Active|
|        588| Germany_Male|0.96878695|                4|        Inactive|
|        593|France_Female|  4.859431|                8|        Inactive|
|        678|   Spain_Male| 1.2959695|                1|        Inactive|
|        676|  France_Male|       0.0|                8|        Inactive|
|        583| Germany_Male|0.47572505|                4|          Active|
+-----------+-------------+----------+

In [62]:
data_updated = (
    data
    .withColumn("GeoGender",
                geo_gender_udf(col("Geography"), col("Gender")))
    .withColumn("BE_Ratio",
                balance_salary_ratio_udf(col("Balance"), col("EstimatedSalary")))
    .withColumn("TotalProductsUsed",
                total_products_used_udf(col("NumOfProducts"), col("Tenure")))
    .withColumn("CardActiveMember",
                card_active_member_udf(col("HasCrCard"), col("IsActiveMember")))
)

In [64]:
data_updated.printSchema()

root
 |-- id: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: double (nullable = true)
 |-- IsActiveMember: double (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)
 |-- GeoGender: string (nullable = true)
 |-- BE_Ratio: float (nullable = true)
 |-- TotalProductsUsed: integer (nullable = true)
 |-- CardActiveMember: string (nullable = true)



In [65]:
data_updated.createOrReplaceTempView("customer_data")

In [69]:
spark.sql("""
SELECT GeoGender, EstimatedSalary, TotalProductsUsed, CardActiveMember, Exited
FROM customer_data
LIMIT 10
""").show()

+-------------+---------------+-----------------+----------------+------+
|    GeoGender|EstimatedSalary|TotalProductsUsed|CardActiveMember|Exited|
+-------------+---------------+-----------------+----------------+------+
|  France_Male|      181449.97|                6|        Inactive|     0|
|  France_Male|        49503.5|                2|          Active|     0|
|  France_Male|      184866.69|               20|        Inactive|     0|
|  France_Male|       84560.88|                2|          Active|     0|
|   Spain_Male|       15068.83|               10|          Active|     0|
| Germany_Male|      136024.31|                4|        Inactive|     1|
|France_Female|       29792.11|                8|        Inactive|     0|
|   Spain_Male|       106851.6|                1|        Inactive|     0|
|  France_Male|      142917.13|                8|        Inactive|     0|
| Germany_Male|      170843.07|                4|          Active|     0|
+-------------+---------------+-------